# 08 — context-dependent DE comparison - do changes induced by CRISPR-KO differ in different culture contexts? (perturbation_2)

In [ ]:
# =============================================================================
# nb08 — Context dependence: the project's core question
#
# Which perturbations behave DIFFERENTLY across immune conditions? A knockout
# that does the same thing in all three arms is constitutive and uninteresting
# here. The signal is perturbations whose signature DIVERGES across conditions
# — that divergence IS the gene x environment interaction the project exists
# to find.
#
# Two independent readouts, deliberately not one:
#
#   1. Signature divergence. For each perturbation, correlate its log2FC
#      signature across conditions. High correlation = does the same thing
#      everywhere (B2M-like). Low = redirected by environment (STAT1-like).
#
#   2. E-distance. A model-free, permutation-tested effect size: did this
#      perturbation do ANYTHING real in this condition, independent of the
#      DESeq2 model? This is the honest significance test — it does not inherit
#      the anti-conservative p-values from the pseudo-replication structure.
#
# A perturbation is called context-dependent only if BOTH hold: a real effect
# somewhere (E-distance), AND divergence across conditions (low correlation).
# Requiring only divergence would fill the hit list with noise, since two null
# signatures are also uncorrelated.
# =============================================================================

%load_ext autoreload
%autoreload 2

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text

from src.config import load_config, load_panels, paths, set_seed
from src.plotting import apply_style, condition_palette, savefig
from src.stats import cross_condition_correlation

cfg    = load_config()
panels = load_panels()
P      = paths(cfg)
SEED   = set_seed(cfg)
apply_style(cfg)
sc.settings.verbosity = 1

s          = cfg["schema"]["obs"]
PERT       = s["perturbation"]
COND       = s["condition"]
CTRL       = cfg["schema"]["control_label"]
REF        = cfg["schema"]["conditions"]["reference"]   # "Control"
cond_order = ["Control", "IFNγ", "Co-culture"]
pal        = condition_palette(cfg)

# ---- signature matrices ---------------------------------------------------
sig      = pd.read_parquet(P.data_processed / "05_signatures_lfc.parquet")
padj     = pd.read_parquet(P.data_processed / "05_signatures_padj.parquet")
adt_sig  = pd.read_parquet(P.data_processed / "07_adt_signatures.parquet")

# ---- QC flags carried forward ---------------------------------------------
flags = pd.read_csv(P.tables / "02_gene_flags.csv", index_col=0)   # high guide spread

print(f"RNA signature: {sig.shape[0]} contrasts x {sig.shape[1]} genes")
print(f"ADT signature: {adt_sig.shape[0]} contrasts x {adt_sig.shape[1]} features")
print(f"\nperturbations: {sig.index.get_level_values(0).nunique()}")
print(f"conditions:    {sorted(sig.index.get_level_values(1).unique())}")
print(f"high guide-spread flags: {int(flags['high_spread'].sum())}")

# for E-distance we need the cells themselves, in PCA space
import mudata as md
mdata = md.read(P.data_interim / "frangieh_qc.h5mu")
rna = mdata["rna"]
emb = pd.read_parquet(P.data_interim / "03_embedding.parquet")
tcell = emb.index[emb["cluster"].astype(str) == "12"]
rna = rna[~rna.obs_names.isin(tcell)].copy()
print(f"\ncells for E-distance: {rna.n_obs:,}")

In [ ]:
# ---- signature divergence across conditions -------------------------------
# For each perturbation, correlate its log2FC signature in IFN-γ and in
# Co-culture against its signature in Control. Low correlation = the effect
# is redirected by the environment. Magnitude ratio = is the same-direction
# effect amplified or damped.
from src.stats import cross_condition_correlation

corr = cross_condition_correlation(sig, reference=REF)
print(f"{len(corr)} (perturbation, condition) comparisons")
print(corr.sort_values("pearson_vs_reference").head(15).round(3).to_string(index=False))

In [ ]:
# ---- quick look before adding E-distance ----------------------------------
# Two axes of context dependence: redirection (x) and amplification (y).
# STAT1 should sit low-left, high-up. B2M near x=1, y=0 — a built-in control.
fig, ax = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

LABEL_MAG = 0.25   # also label anything with |log2 magnitude ratio| above this
anchors = ["B2M", "STAT1", "JAK1", "JAK2", "IFNGR1", "IFNGR2", "CD58", "CD9"]

for a, cond in zip(ax, ["IFNγ", "Co-culture"]):
    d = corr[corr["condition"] == cond].copy()
    d["log2_mag"] = np.log2(d["magnitude_ratio"])

    a.scatter(d["pearson_vs_reference"], d["log2_mag"],
              s=22, alpha=0.5, color=pal.get(cond, "#888"),
              edgecolor="none", rasterized=True, zorder=2)

    a.axvline(cfg["context"]["call_thresholds"]["max_cross_condition_pearson"],
              ls="--", c="crimson", lw=1, label="divergence threshold")
    a.axhline(0, ls=":", c="k", lw=0.8)

    # sparse: 5 most redirected + 4 most amplified, PLUS |log2 mag| > LABEL_MAG,
    # PLUS the named mechanistic anchors
    to_label = pd.concat([
        d.nsmallest(5, "pearson_vs_reference"),
        d.nlargest(4, "magnitude_ratio"),
        d[d["perturbation"].isin(anchors)],
    ]).drop_duplicates("perturbation")

    texts = []
    for _, r in to_label.iterrows():
        is_anchor = r["perturbation"] in anchors
        a.scatter(r["pearson_vs_reference"], r["log2_mag"], s=44, zorder=5,
                  color="black" if is_anchor else pal.get(cond, "#888"),
                  edgecolor="white", linewidth=0.6)
        texts.append(a.text(r["pearson_vs_reference"], r["log2_mag"],
                            r["perturbation"], fontsize=6.5,
                            fontweight="bold" if is_anchor else "normal",
                            zorder=6))

    adjust_text(texts, ax=a,
                arrowprops=dict(arrowstyle="-", lw=0.35, color="grey"),
                expand=(1.25, 1.4))

    a.set_xlabel("Pearson r vs Control signature\n(low = redirected)")
    a.set_ylabel("log2 magnitude ratio\n(>0 = amplified in this condition)")
    a.set_title(f"{cond} vs Control", fontweight="bold", fontsize=12)
    a.legend(fontsize=8, loc="lower left")
    print(f"{cond}: {len(to_label)} labelled")

fig.suptitle("Signature divergence across conditions\n"
             "(black = mechanistic anchors)",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
savefig(fig, "08_divergence", cfg)

In [ ]:
d = corr[corr["condition"] == "IFNγ"].copy()
print("correlation distribution:")
print(d["pearson_vs_reference"].describe().round(3))
print(f"\nl2_reference median: {d['l2_reference'].median():.2f}")
print(f"\nB2M specifically:")
print(d[d["perturbation"] == "B2M"][
    ["pearson_vs_reference", "l2_reference", "magnitude_ratio"]].round(3).to_string(index=False))
print(f"\ntop 15 by correlation:")
print(d.nlargest(15, "pearson_vs_reference")[
    ["perturbation", "pearson_vs_reference", "l2_reference"]].round(3).to_string(index=False))

In [ ]:
def corr_on_responsive(sig, padj, reference, alpha=0.05, min_genes=15):
    """Cross-condition correlation on genes significant in either condition,
    per perturbation — not on all 5000, where noise dominates."""
    out = []
    perts = sig.index.get_level_values("perturbation").unique()
    for pert in perts:
        if (pert, reference) not in sig.index:
            continue
        ref_sig = sig.loc[(pert, reference)]
        ref_p   = padj.loc[(pert, reference)]
        for cond in sig.index.get_level_values("condition").unique():
            if cond == reference or (pert, cond) not in sig.index:
                continue
            cond_sig = sig.loc[(pert, cond)]
            cond_p   = padj.loc[(pert, cond)]
            # genes responsive in EITHER condition
            responsive = (ref_p < alpha) | (cond_p < alpha)
            n = int(responsive.sum())
            if n < min_genes:
                out.append({"perturbation": pert, "condition": cond,
                            "pearson_vs_reference": np.nan, "n_responsive": n,
                            "l2_reference": float(np.linalg.norm(ref_sig)),
                            "magnitude_ratio": np.nan})
                continue
            r = np.corrcoef(ref_sig[responsive], cond_sig[responsive])[0, 1]
            out.append({
                "perturbation": pert, "condition": cond,
                "pearson_vs_reference": r, "n_responsive": n,
                "l2_reference": float(np.linalg.norm(ref_sig)),
                "l2_condition": float(np.linalg.norm(cond_sig)),
                "magnitude_ratio": float(np.linalg.norm(cond_sig) /
                                         (np.linalg.norm(ref_sig) + 1e-9)),
            })
    return pd.DataFrame(out)

corr = corr_on_responsive(sig, padj, reference=REF, alpha=0.05)
d = corr[corr["condition"] == "IFNγ"]
print(d["pearson_vs_reference"].describe().round(3))
print(f"\nperturbations with >=15 responsive genes: {d['pearson_vs_reference'].notna().sum()}")
print(f"\nB2M:")
print(corr[corr['perturbation']=='B2M'][['condition','pearson_vs_reference','n_responsive','magnitude_ratio']].round(3).to_string(index=False))